# Python pour l'Ingénierie des Matériaux
## Séance 3 — Feuille d'exercices — SOLUTIONS

**Conservatoire National des Arts et Métiers**  
*04 Mars 2026*

---

Cette feuille d'exercices accompagne la séance 3. Elle est divisée en cinq parties :

| Partie | Thème |
|--------|-------|
| 1 | Tableaux NumPy — création et attributs |
| 2 | Opérations vectorisées et agrégations |
| 3 | Indexation, découpage et masques booléens |
| 4 | Lecture de fichiers avec NumPy + graphiques |
| 5 | Mini-projet guidé — Arrhenius sur une grille |

> **Conseil :** Lisez l'énoncé entier avant d'écrire la moindre ligne de code.  
> Ajoutez des commentaires (`#`) pour expliquer votre raisonnement.  
> Dans cette séance, la règle est : **aucune boucle `for` ou `while` sur les éléments d'un tableau NumPy** — utilisez les opérations vectorisées.

---
## Partie 1 — Tableaux NumPy : création et attributs

### Exercice 1.1 — Créer et inspecter des tableaux

Créez les tableaux suivants **en utilisant la fonction appropriée**, puis affichez la valeur, le `shape`, le `dtype` et le `ndim` de chacun :

| Tableau | Fonction suggérée | Contenu attendu |
|---------|-------------------|-----------------|
| `T_rampe` | `np.linspace` | 10 températures de 20 à 1100 °C inclus |
| `eps_vals` | `np.arange` | Déformations de 0.000 à 0.020 (exclus) avec un pas de 0.002 |
| `zeros_5` | `np.zeros` | Tableau de 5 zéros |
| `uns_6` | `np.ones` | Tableau de 6 uns |
| `mat_prop` | `np.array` | Tableau 2D à partir de la liste ci-dessous |

```python
# Données pour mat_prop :
# Ligne 0 : [E (GPa), Rm (MPa), A (%)]
# Acier S355  : E=210, Rm=510, A=22
# Alu 7075-T6 : E= 71, Rm=572, A=11
# Ti-6Al-4V   : E=114, Rm=950, A=10
```

In [ ]:
# Exercice 1.1
import numpy as np

# ── Rampe de température ─────────────────────────────────────────────────
T_rampe = np.linspace(20, 1100, 10)
print("T_rampe :", T_rampe)
print("shape :", T_rampe.shape, "  dtype :", T_rampe.dtype, "  ndim :", T_rampe.ndim)

# ── Déformations ─────────────────────────────────────────────────────────
eps_vals = np.arange(0.000, 0.020, 0.002)
print("eps_vals :", eps_vals)
print("shape :", eps_vals.shape, "  dtype :", eps_vals.dtype)

# ── Tableaux de zéros et de uns ──────────────────────────────────────────
zeros_5 = np.zeros(5)
uns_6   = np.ones(6)
print("zeros_5 :", zeros_5)
print("uns_6   :", uns_6)

# ── Tableau 2D des propriétés matériaux ─────────────────────────────────
mat_prop = np.array([
    [210, 510, 22],   # Acier S355
    [ 71, 572, 11],   # Alu 7075-T6
    [114, 950, 10],   # Ti-6Al-4V
])
print("mat_prop :", mat_prop)
print("shape :", mat_prop.shape, "  dtype :", mat_prop.dtype)

*Questions :*

1. Quel est le `dtype` de `mat_prop` et pourquoi ?

   Le `dtype` est `int64`, car toutes les valeurs fournies dans la liste Python sont des entiers (210, 510, 22, etc.). NumPy infère automatiquement le type de données lors de la création du tableau avec `np.array()`.

2. Pourquoi `T_rampe` a-t-il exactement 10 éléments alors que vous avez passé `10` comme troisième argument de `linspace` ?

   Le troisième argument de `np.linspace(a, b, n)` est le nombre de points à générer. NumPy place exactement `n` points régulièrement espacés entre `a` et `b` (incluses).

### Exercice 1.2 — Indexation 2D et extraction de colonnes

Reprenez le tableau `mat_prop` de l'exercice 1.1.

**Sans boucle**, extrayez :

1. La **deuxième ligne** entière (les propriétés de l'aluminium).
2. La **première colonne** entière (les modules de Young).
3. La valeur à la **ligne 2, colonne 1** (Rm du Ti-6Al-4V).
4. Un **sous-tableau** contenant les deux premières lignes et les deux premières colonnes.

Vérifiez numériquement que `mat_prop[1, :]` et `mat_prop[1]` donnent le même résultat.

*Rappel :* la syntaxe est `tableau[ligne, colonne]` — une seule paire de crochets avec une virgule.

In [ ]:
# Exercice 1.2
# (mat_prop doit déjà être défini dans la cellule précédente)

# 1. Deuxième ligne (indice 1 — aluminium 7075-T6)
ligne_alu = mat_prop[1, :]
print("1. Ligne aluminium :", ligne_alu)

# 2. Première colonne — modules de Young
modules_E = mat_prop[:, 0]
print("2. Modules E (GPa) :", modules_E)

# 3. Rm du Ti-6Al-4V (ligne 2, colonne 1)
Rm_Ti = mat_prop[2, 1]
print("3. Rm Ti-6Al-4V    :", Rm_Ti, "MPa")

# 4. Sous-tableau 2×2 (2 premières lignes, 2 premières colonnes)
sous_tableau = mat_prop[0:2, 0:2]
print("4. Sous-tableau 2x2 :\n", sous_tableau)

# Vérification — les deux syntaxes donnent le même résultat
print("mat_prop[1, :] == mat_prop[1] :", np.array_equal(mat_prop[1, :], mat_prop[1]))

---
## Partie 2 — Opérations vectorisées et agrégations

### Exercice 2.1 — Calcul vectorisé : volumes de cylindres

Un laboratoire teste **8 éprouvettes cylindriques**. Les dimensions mesurées sont :

```python
diametres = np.array([10.02, 9.98, 10.05, 9.97, 10.01, 9.99, 10.03, 10.00])  # mm
longueurs = np.array([50.1,  49.8, 50.3,  50.0, 49.9,  50.2, 50.0,  50.1 ])  # mm
```

**Sans boucle**, calculez pour chaque éprouvette :

1. L'**aire de section** $A = \pi (d/2)^2$ en mm².
2. Le **volume** $V = A \times L$ en mm³.
3. La **masse** $m = \rho V$ en grammes, avec $\rho = 7850 \times 10^{-3}$ g/mm³ (acier).

Puis calculez et affichez :
- La masse **moyenne**, l'**écart-type** et les valeurs **min/max**.
- Le **coefficient de variation** $CV = \sigma / \bar{m} \times 100$ % — indicateur de la dispersion relative.

In [ ]:
# Exercice 2.1
import numpy as np

diametres = np.array([10.02, 9.98, 10.05, 9.97, 10.01, 9.99, 10.03, 10.00])  # mm
longueurs = np.array([50.1,  49.8, 50.3,  50.0, 49.9,  50.2, 50.0,  50.1 ])  # mm
rho = 7850e-3   # g/mm³

# 1. Aires de section (mm²)
aires = np.pi * (diametres / 2) ** 2
print("Aires (mm2) :", np.round(aires, 3))

# 2. Volumes (mm³)
volumes = aires * longueurs
print("Volumes (mm3) :", np.round(volumes, 1))

# 3. Masses (g)
masses = rho * volumes
print("Masses  (g) :", np.round(masses, 2))

# Statistiques
print("Moyenne :", round(np.mean(masses), 3), "g")
print("Ecart-type:", round(np.std(masses), 4), "g")
print("Min / Max:", round(np.min(masses), 3), "/", round(np.max(masses), 3), "g")

# Coefficient de variation
CV = np.std(masses) / np.mean(masses) * 100
print("CV :", round(CV, 4), "%")

### Exercice 2.2 — Agrégations 2D avec `axis=`

Le tableau ci-dessous représente les mesures de trois propriétés mécaniques de **5 lots d'acier** :

```python
# Colonnes : [UTS (MPa), limite d'élasticité Re (MPa), allongement A (%)]
lots = np.array([
    [510, 355, 22],
    [495, 340, 24],
    [525, 368, 19],
    [502, 350, 21],
    [518, 362, 23],
])
```

**Sans boucle**, calculez :

1. La **moyenne de chaque propriété** (moyenne sur les 5 lots) — résultat de taille `(3,)`.
2. La **moyenne de chaque lot** (moyenne de ses 3 propriétés) — interprétation à relativiser (les unités diffèrent !).
3. L'**écart-type de chaque propriété**.
4. La valeur **maximale** et **minimale** de l'UTS (première colonne).

> *Rappel :* `axis=0` opère « vers le bas » (réduit les lignes), `axis=1` opère « vers la droite » (réduit les colonnes).

In [ ]:
# Exercice 2.2
import numpy as np

lots = np.array([
    [510, 355, 22],
    [495, 340, 24],
    [525, 368, 19],
    [502, 350, 21],
    [518, 362, 23],
])

# 1. Moyenne de chaque propriété (axis=0)
moy_proprietes = np.mean(lots, axis=0)
print("1. Moyennes [UTS, Re, A] :", np.round(moy_proprietes, 2))

# 2. Moyenne de chaque lot (axis=1)
moy_lots = np.mean(lots, axis=1)
print("2. Moyennes par lot      :", np.round(moy_lots, 2))

# 3. Écart-type de chaque propriété
std_proprietes = np.std(lots, axis=0)
print("3. Ecart-types [UTS,Re,A]:", np.round(std_proprietes, 2))

# 4. Max et min de l'UTS
UTS = lots[:, 0]   # première colonne
print("4. UTS max :", np.max(UTS), "MPa   UTS min :", np.min(UTS), "MPa")

---
## Partie 3 — Indexation, découpage et masques booléens

### Exercice 3.1 — Filtrage par masque booléen

Voici les valeurs de résistance à la traction $R_m$ (MPa) mesurées sur 12 éprouvettes :

```python
UTS = np.array([480, 530, 510, 470, 560, 490, 615, 505, 495, 540, 458, 512])
```

Le cahier des charges impose $R_m \geq 500$ MPa.

**En utilisant des masques booléens (aucune boucle)** :

1. Créez le masque `conforme` qui vaut `True` pour les éprouvettes passant le critère.
2. Extrayez les valeurs **conformes** et les valeurs **non conformes** dans deux tableaux.
3. Calculez le **taux de conformité** (%) sans boucle.
4. Calculez la **valeur moyenne** des éprouvettes conformes.
5. Utilisez `np.where()` pour créer un tableau `statut` contenant la chaîne `"OK"` ou `"NON"` pour chaque éprouvette.

In [ ]:
# Exercice 3.1
import numpy as np

UTS = np.array([480, 530, 510, 470, 560, 490, 615, 505, 495, 540, 458, 512])

# 1. Masque — True pour les éprouvettes conformes (UTS >= 500 MPa)
conforme = UTS >= 500
print("Masque conforme :", conforme)

# 2. Valeurs conformes et non conformes
valeurs_conformes     = UTS[conforme]
valeurs_non_conformes = UTS[~conforme]
print("Conformes :", valeurs_conformes)
print("Non conformes :", valeurs_non_conformes)

# 3. Taux de conformité
taux = np.sum(conforme) / len(UTS) * 100
print("3. Taux de conformite :", round(taux, 1), "%")

# 4. Moyenne des conformes
moy_conformes = np.mean(valeurs_conformes)
print("4. Moyenne conformes :", round(moy_conformes, 1), "MPa")

# 5. np.where — "OK" si conforme, "NON" sinon
statut = np.where(conforme, "OK", "NON")
print("5. Statut :", statut)

### Exercice 3.2 — Masques multi-critères sur un tableau 2D

Le tableau `specimens` ci-dessous contient pour chaque spécimen :  
**colonne 0 :** UTS (MPa) | **colonne 1 :** allongement à rupture (%) | **colonne 2 :** dureté HB

```python
specimens = np.array([
    [480, 14, 235],
    [530, 11, 258],
    [510, 16, 245],
    [470,  9, 221],
    [560, 18, 265],
    [490, 13, 238],
    [615,  8, 290],
    [505, 15, 248],
])
```

**Critères de conformité :**
- UTS ≥ 495 MPa
- Allongement ≥ 12 %
- Dureté entre 230 et 270 HB (inclus)

1. Créez un masque booléen combinant les trois critères (opérateur `&`).
2. Extrayez les **lignes conformes** (les spécimens qui passent tous les critères).
3. Comptez les spécimens échouant à **chaque critère individuellement**.
4. Affichez les **indices** des spécimens non conformes avec `np.where()`.

In [ ]:
# Exercice 3.2
import numpy as np

specimens = np.array([
    [480, 14, 235],
    [530, 11, 258],
    [510, 16, 245],
    [470,  9, 221],
    [560, 18, 265],
    [490, 13, 238],
    [615,  8, 290],
    [505, 15, 248],
])

UTS  = specimens[:, 0]
allo = specimens[:, 1]
HB   = specimens[:, 2]

# 1. Masque combiné
# Attention : utiliser & (ET bit-à-bit), PAS and
masque_UTS   = UTS >= 495
masque_allo  = allo >= 12
masque_HB    = (HB >= 230) & (HB <= 270)
masque_total = masque_UTS & masque_allo & masque_HB
print("1. Masque total :", masque_total)

# 2. Lignes conformes
conformes = specimens[masque_total]
print("2. Specimens conformes :", conformes)

# 3. Nombre d'échecs par critère
print("3. Echecs UTS  :", np.sum(~masque_UTS))
print("   Echecs allo :", np.sum(~masque_allo))
print("   Echecs HB   :", np.sum(~masque_HB))

# 4. Indices non conformes
indices_nc = np.where(~masque_total)
print("4. Indices non conformes :", indices_nc[0])

---
## Partie 4 — Lecture de fichiers avec NumPy + graphiques

### Exercice 4.1 — Charger un CSV avec `np.loadtxt`

Le fichier `donnees_traction.csv` contient les résultats d'un essai de traction sur un acier 316L.  
Les deux premières lignes sont des commentaires (caractère `#`), puis vient la ligne d'en-tête, puis les données.

1. Chargez le fichier en **une seule ligne** avec `np.loadtxt()`, en ignorant les commentaires et l'en-tête.
2. Extrayez les colonnes `deformations` et `contraintes` par indexation 2D.
3. Calculez et affichez :
   - Le nombre de points de mesure.
   - La déformation maximale.
   - La contrainte maximale (= UTS expérimentale).
   - La contrainte moyenne et l'écart-type avec `np.mean()` et `np.std()`.

> *Rappel de la structure du fichier :*
> ```
> # Essai de traction — acier 316L
> # Colonnes : deformation (sans unite), contrainte (MPa)
> deformation,contrainte
> 0.0000,0.0
> 0.0005,105.0
> ...
> ```

In [ ]:
# Exercice 4.1
import numpy as np

# 1. Charger le fichier
# skiprows=3 : 2 lignes de commentaires (#) + 1 ligne d'en-tête à ignorer
data = np.loadtxt("donnees_traction.csv", delimiter=",", skiprows=3)
print("Tableau chargé — shape :", data.shape)

# 2. Extraire les colonnes par indexation 2D
deformations = data[:, 0]
contraintes  = data[:, 1]

# 3. Statistiques
print("Nb de points    :", len(deformations))
print("Deformation max :", round(np.max(deformations), 4))
print("Contrainte max  :", round(np.max(contraintes), 1), "MPa")
print("Contrainte moy  :", round(np.mean(contraintes), 1), "MPa")
print("Ecart-type      :", round(np.std(contraintes), 2), "MPa")

### Exercice 4.2 — Tracé de la courbe contrainte–déformation

En utilisant les tableaux `deformations` et `contraintes` de l'exercice 4.1, tracez la **courbe contrainte–déformation** et répondez aux questions dans la cellule Markdown suivante.

**Exigences du graphique :**

1. Axe x : déformation $\varepsilon$, fontsize 13.
2. Axe y : contrainte $\sigma$ (MPa), fontsize 13.
3. Titre : `"Courbe contrainte-deformation — acier 316L"`, fontsize 14.
4. Courbe en `'steelblue'`, épaisseur 2, marqueurs `'o'` taille 4.
5. Grille et légende activées.

**En plus :** ajoutez sur le même graphique une ligne verticale `plt.axvline()` à la déformation qui correspond à la **contrainte maximale** (utilisez `np.argmax()` pour trouver l'indice).

> `np.argmax(a)` — *argument of the maximum* — retourne l'**indice** de la valeur maximale du tableau `a`.

In [ ]:
# Exercice 4.2
import numpy as np
import matplotlib.pyplot as plt

# (deformations et contraintes doivent être définis dans la cellule précédente)

# Indice de la contrainte maximale et déformation correspondante
idx_max = np.argmax(contraintes)
eps_UTS = deformations[idx_max]   # déformation à l'UTS
print("UTS max :", round(contraintes[idx_max], 1), "MPa")
print("Deformation a l'UTS :", round(eps_UTS, 4))

plt.figure(figsize=(8, 5))

# Tracé de la courbe contrainte–déformation
plt.plot(deformations, contraintes,
         color='steelblue', linewidth=2, marker='o', markersize=4,
         label='Acier 316L')

# Ligne verticale à la déformation correspondant à l'UTS
plt.axvline(x=eps_UTS,
            color='tomato', linestyle='--', linewidth=1.5,
            label='Deformation a l\'UTS')  # \' est utilisé pour "échapper" l'apostrophe dans la chaîne de caractères (sinon il y aurait une erreur de syntaxe)

plt.xlabel("Deformation", fontsize=13)
plt.ylabel("Contrainte (MPa)", fontsize=13)
plt.title("Courbe contrainte-deformation — acier 316L", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

**Vos réponses ici :**

1. UTS mesurée : 538.0 MPa
2. Déformation à l'UTS : 0.15
3. Décrivez en une phrase l'allure de la courbe (domaine élastique, domaine plastique, striction) : La courbe présente un domaine élastique linéaire jusqu'à ε ≈ 0.002, puis un long domaine plastique avec écrouissage progressif jusqu'à l'UTS (ε = 0.15, σ = 538 MPa), suivi d'une une chute de la contrainte jusqu'à la rupture à ε = 0.20.

---
## Partie 5 — Mini-projet guidé : Carte d'Arrhenius

### Scénario

Vous étudiez une réaction d'oxydation dont la cinétique suit la loi d'Arrhenius :

$$k(T, E_a) = A_0 \, \exp\!\left(-\frac{E_a}{RT}\right)$$

avec $R = 8{,}314$ J/(mol·K) et $A_0 = 10^{12}$.

Votre mission est de construire une **carte de la constante cinétique** en fonction de la température ($300$–$1200$ K) et de l'énergie d'activation ($50$–$300$ kJ/mol), puis d'analyser les résultats.

**Étapes :**

1. Construire les vecteurs 1D de températures et d'énergies d'activation.
2. Créer la grille 2D avec `np.meshgrid()`.
3. Calculer $k$ sur la grille (en une seule instruction vectorisée).
4. Tracer la carte en contours remplis (`contourf`) de $\log_{10}(k)$ avec une labellisaiton correcte.
5. Tracer une carte de chaleur (`pcolormesh`) en parallèle (pareil).
6. Répondre aux questions d'analyse dans la cellule Markdown finale.

In [ ]:
# ── Étape 1 & 2 : Vecteurs et grille ───
import numpy as np

R  = 8.314  # J/(mol·K)
A0 = 1e12   # facteur pré-exponentiel

# Vecteurs 1D
T_vec  = np.linspace(300, 1200, 100)    # températures 300–1200 K, 100 points
Ea_vec = np.linspace(50e3, 300e3, 100)  # Ea 50 000–300 000 J/mol, 100 points

# Grille 2D — np.meshgrid retourne (T_grid, Ea_grid)
T_grid, Ea_grid = np.meshgrid(T_vec, Ea_vec)

print("T_grid  shape :", T_grid.shape)
print("Ea_grid shape :", Ea_grid.shape)

In [ ]:
# ── Étape 3 : Calcul vectorisé de k ───
k_grid = A0 * np.exp(-Ea_grid / (R * T_grid))

print("k_grid shape :", k_grid.shape)
print("k minimum    :", round(np.min(k_grid), 6))
print("k maximum    :", round(np.max(k_grid), 2))

In [ ]:
# ── Étape 4 : Contours remplis de log10(k) ──
import matplotlib.pyplot as plt

log_k = np.log10(k_grid)  # log base 10

plt.figure(figsize=(8, 5))

# Tracé des contours remplis de log10(k) en fonction de T et Ea
cf = plt.contourf(T_grid, Ea_grid / 1000, log_k,
                  levels=30,
                  cmap='viridis')
plt.colorbar(cf, label='log10(k)')
plt.xlabel("Temperature T (K)", fontsize=12)
plt.ylabel("Energie d\'activation Ea (kJ/mol)", fontsize=12)  # Encore l'apostrophe est échappée avec \'
plt.title("Constante cinematique d\'Arrhenius — log10(k)", fontsize=13)  # Pareil
plt.tight_layout()
plt.show()

In [ ]:
# ── Étape 5 : Carte de chaleur pcolormesh ──
plt.figure(figsize=(8, 5))

# Carte de chaleur de log10(k) en fonction de T et Ea
pc = plt.pcolormesh(T_grid, Ea_grid / 1000, log_k,
                    cmap='viridis', shading='auto')
plt.colorbar(pc, label='log10(k)')
plt.xlabel("Temperature T (K)", fontsize=12)
plt.ylabel("Energie d\'activation Ea (kJ/mol)", fontsize=12)  # Pareil
plt.title("Constante cinematique d\'Arrhenius — log10(k)", fontsize=13)  # Pareil
plt.tight_layout()
plt.show()

### Étape 6 — Analyse d'ingénierie

*Répondez aux questions suivantes en vous appuyant sur les cartes tracées ci-dessus.*

---

**1. Où la réaction devient-elle « rapide » sur la carte ?**
*(En termes de température et d'énergie d'activation)*

La réaction est rapide pour des températures élevées (> 900 K) et de faibles énergies d'activation (< 100 kJ/mol) — c'est la région en bas à droite de la carte (haute T, faible Ea).

---

**2. Pour une énergie d'activation fixe de 150 kJ/mol, quel facteur d'augmentation de k observe-t-on entre 400 K et 1000 K ?**
*(Utilisez la cellule de code ci-dessous pour calculer.)*

On obtient un facteur d'environ 5.7 × 10¹¹ : la cinétique est très sensible à la température, illustrant la nature exponentielle de la loi d'Arrhenius.

---

**3. Pourquoi traceons-nous log₁₀(k) plutôt que k directement ?**

k varie sur de très nombreux ordres de grandeur (de ~10⁻²⁰ à ~10¹²). Tracer k directement écraserait toutes les petites valeurs et rendrait le graphique illisible. log₁₀(k) comprime cette dynamique sur une échelle linéaire exploitable.

---

**4. Quelle est la différence visuelle entre `contourf` et `pcolormesh` ? Pour un rapport d'ingénierie, lequel préféreriez-vous et pourquoi ?**

`contourf` trace des iso-courbes régulières qui facilitent la lecture des seuils de valeurs précis (ex. T minimale pour atteindre log10(k) > −5). `pcolormesh` colore chaque cellule de la grille indépendamment, donnant une vue plus brute de la distribution. Pour un rapport d'ingénierie, `contourf` est préférable car il met en évidence les lignes d'iso-constante cinétique et permet une lecture directe des conditions de traitement thermique, qui facilite de son tour la conception.

In [ ]:
# ── Calculs pour la question 2 ──
import numpy as np

R  = 8.314
A0 = 1e12
Ea = 150e3  # J/mol — énergie d'activation fixe

k_400  = A0 * np.exp(-Ea / (R * 400))   # k à 400 K
k_1000 = A0 * np.exp(-Ea / (R * 1000))  # k à 1000 K
facteur = k_1000 / k_400

print("k(400 K)  =", k_400)
print("k(1000 K) =", k_1000)
print("Facteur k(1000)/k(400) =", round(facteur, 2))